In [1]:
# R2_se_alignment.ipynb  (revision stage 2 - inference, addresses M4)
# Estimand-matched SE comparison in ONE coordinate (per-bin log-odds contribution),
# both models MULTIVARIATE on the SAME fixed bin grid (grid-conditional SE, stated):
#   free model     : dummy-logit  gamma_hat_j(b)        (penalized, near-MLE)
#   rank-one model : logistic-WoE beta_hat_j*WoE_j(b)
# SE is estimated by a shared fair-coordinate BOOTSTRAP (not the information matrix,
# which diverges under separation in extreme credit bins). Each replicate recomputes
# WoE on the fixed grid and refits BOTH models, so both absorb the same noise.
# We report (i) raw ratio, (ii) beta-corrected ratio (reviewer's 1/beta hypothesis),
# (iii) TOST equivalence [0.8,1.25], (iv) bootstrap CI. Stratified by strength & missing.
# Single-cell; greyscale seaborn; dpi600 png+pdf; no captions.

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
import statsmodels.api as sm
warnings.filterwarnings('ignore')

SEED=42; N_BOOT=200; TOST_LO=0.8; TOST_HI=1.25
rng=np.random.default_rng(SEED)
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})
ROOT=os.path.abspath(os.path.join(os.getcwd(),'..')); DATA=os.path.join(ROOT,'data')
TABLES=os.path.join(ROOT,'results','tables'); FIGS=os.path.join(ROOT,'results','figures')
def savefig(fig,name):
    for e in ('png','pdf'): fig.savefig(os.path.join(FIGS,name+'.'+e),dpi=600,bbox_inches='tight')
    plt.close(fig)

# fixed grid from stage-0 assets (grid-conditional SE; M6 leakage handled in stages 1&3)
woe_X=pd.read_parquet(os.path.join(DATA,'gmsc_woe.parquet'))
dfc=pd.read_parquet(os.path.join(DATA,'gmsc_clean.parquet'))
meta=json.load(open(os.path.join(DATA,'woe_maps.json')))
TARGET=meta['target']; FEATURES=meta['features']; MAPS=meta['maps']
y=woe_X[TARGET].values.astype(int)
bo={}; bin_w={}; nbk={}; neutral={}
for c in FEATURES:
    recs=sorted([r for r in MAPS[c] if r['bin']>=0],key=lambda r:r['lo'])+[r for r in MAPS[c] if r['bin']==-1]
    bo[c]=recs; bin_w[c]=np.array([r['n'] for r in recs],float); nbk[c]=len(recs)
    neutral[c]=np.array([abs(r['woe'])<1e-9 for r in recs])
def tobin(x,recs):
    idx=np.full(len(x),len(recs)-1,int); fin=[r for r in recs if r['bin']>=0]
    for p,r in enumerate(fin):
        sel=(~x.isna()).values&(x>r['lo']).values&(x<=r['hi']).values; idx[sel]=p
    if any(r['bin']==-1 for r in recs): idx[x.isna().values]=len(recs)-1
    else: idx[x.isna().values]=0
    return idx
Xbin={c:tobin(dfc[c],bo[c]) for c in FEATURES}
has_miss={c:any((r['bin']==-1 and abs(r['woe'])>1e-9) for r in MAPS[c]) for c in FEATURES}

# point-estimate beta (for strength label and beta-correction)
Xw0=np.column_stack([np.array([r['woe'] for r in bo[c]])[Xbin[c]] for c in FEATURES])
beta0={c:sm.Logit(y,sm.add_constant(Xw0)).fit(disp=0).params[i+1] for i,c in enumerate(FEATURES)}

def woe_from_counts(bidx,yb,nb,tg,tb,neu):
    g=np.zeros(nb); b=np.zeros(nb); np.add.at(g,bidx[yb==0],1.0); np.add.at(b,bidx[yb==1],1.0)
    w=np.log((b+0.5)/(tb+0.5))-np.log((g+0.5)/(tg+0.5)); w[neu]=0.0; return w

n=len(y)
boot_woe_contrib={c:[] for c in FEATURES}   # beta^(r)*WoE^(r)(b)
boot_gam_contrib={c:[] for c in FEATURES}   # gamma_hat^(r)(b)
t0=time.time()
for r in range(N_BOOT):
    idx=rng.integers(0,n,n); yb=y[idx]; tg=int((yb==0).sum()); tb=int((yb==1).sum())
    # recompute WoE, WoE-logistic
    woe_b={}; cols=[]
    for c in FEATURES:
        bi=Xbin[c][idx]; w=woe_from_counts(bi,yb,nbk[c],tg,tb,neutral[c]); woe_b[c]=w; cols.append(w[bi])
    Xwb=np.column_stack(cols)
    try:
        lb=sm.Logit(yb,sm.add_constant(Xwb)).fit(disp=0)
        for i,c in enumerate(FEATURES): boot_woe_contrib[c].append(lb.params[i+1]*woe_b[c])
    except Exception:
        for c in FEATURES: boot_woe_contrib[c].append(np.full(nbk[c],np.nan))
    # free dummy penalized logistic (near-MLE, finite under separation)
    dcols=[]; key=[]
    for c in FEATURES:
        bi=Xbin[c][idx]
        for b in range(nbk[c]-1): dcols.append((bi==b).astype(float)); key.append((c,b))
    Xd=np.column_stack(dcols)
    clf=LogisticRegression(C=2000,max_iter=2000,solver='lbfgs').fit(Xd,yb)
    gamma={c:np.zeros(nbk[c]) for c in FEATURES}
    for (c,b),v in zip(key,clf.coef_.ravel()): gamma[c][b]=v
    for c in FEATURES:                   # center within feature (pop-weighted)
        gamma[c]-=np.sum(bin_w[c]*gamma[c])/bin_w[c].sum()
        boot_gam_contrib[c].append(gamma[c])
    if (r+1)%50==0: print('  bootstrap %d/%d (%.0fs)'%(r+1,N_BOOT,time.time()-t0))
print('bootstrap done %.0fs'%(time.time()-t0))

# SE per bin = bootstrap SD; weighted-RMS per feature over non-neutral bins
rows=[]
for c in FEATURES:
    W=np.array(boot_woe_contrib[c]); G=np.array(boot_gam_contrib[c])
    se_w=np.nanstd(W,0,ddof=1); se_g=np.nanstd(G,0,ddof=1)
    keep=~neutral[c]; wn=np.zeros_like(bin_w[c]); wn[keep]=bin_w[c][keep]/bin_w[c][keep].sum()
    rms_w=float(np.sqrt(np.sum(wn[keep]*se_w[keep]**2))); rms_g=float(np.sqrt(np.sum(wn[keep]*se_g[keep]**2)))
    ratio=rms_g/rms_w if rms_w>0 else np.nan
    strength='strong' if abs(beta0[c])>=0.3 else 'weak'
    rows.append([c,strength,'miss' if has_miss[c] else 'nomiss',beta0[c],rms_w,rms_g,ratio,ratio*abs(beta0[c])])
se=pd.DataFrame(rows,columns=['feature','strength','miss','beta','se_woe','se_gam','ratio','ratio_betacorr'])
print('\n=== SE ratio (free gamma vs rank-one beta*WoE), bootstrap, per feature ===')
print(se.round(3).to_string(index=False))

# per-bin bootstrap for a ratio distribution + TOST (bin-level, strong features)
def tost(vals,lo=TOST_LO,hi=TOST_HI):
    v=np.log(np.asarray(vals)); v=v[np.isfinite(v)]
    if len(v)<3: return np.nan
    m=v.mean(); sdv=v.std(ddof=1)/np.sqrt(len(v))
    return max(stats.t.sf((m-np.log(lo))/sdv,len(v)-1), stats.t.sf((np.log(hi)-m)/sdv,len(v)-1))
print('\n=== TOST equivalence (margin [%.2f,%.2f]) on per-feature SE ratio ==='%(TOST_LO,TOST_HI))
for g,label in [('ratio','raw'),('ratio_betacorr','beta-corrected')]:
    for grp in ['strong','weak']:
        sub=se[se.strength==grp][g].values; sub=sub[np.isfinite(sub)]
        if len(sub)<2: continue
        ci=np.percentile(sub,[2.5,97.5]); p=tost(sub)
        print('  %-14s %-7s: mean=%.3f CI[%.2f,%.2f] TOST p=%.3g %s'%(
            label,grp,np.mean(sub),ci[0],ci[1],(p if p==p else -1),'(equiv)' if (p==p and p<0.05) else ''))

se.round(6).to_csv(os.path.join(TABLES,'R2_se_ratio_feature.csv'),index=False)

# FIGURE: per-feature ratio, raw vs beta-corrected, with equivalence band
fig,ax=plt.subplots(figsize=(7.6,4.8))
order=se.sort_values('beta')['feature'].tolist(); xpos=np.arange(len(order))
raw=[se[se.feature==f]['ratio'].values[0] for f in order]
bc =[se[se.feature==f]['ratio_betacorr'].values[0] for f in order]
ax.axhspan(TOST_LO,TOST_HI,color='0.9',zorder=0,label='equivalence [0.8,1.25]')
ax.axhline(1.0,color='0.0',lw=1,ls='--')
ax.scatter(xpos-0.12,raw,s=55,marker='o',c='0.3',edgecolor='0.0',label='raw ratio')
ax.scatter(xpos+0.12,bc, s=55,marker='^',c='0.65',edgecolor='0.0',label='β-corrected')
ax.set_xticks(xpos); ax.set_xticklabels([f[:14] for f in order],rotation=35,ha='right',fontsize=7)
ax.set_ylabel('SE ratio (free γ / rank-one β·WoE)')
ax.legend(frameon=True,facecolor='white',edgecolor='0.5',fontsize=8); savefig(fig,'R2_se_ratio_box')

print('\nSaved: R2_se_ratio_box, R2_se_ratio_feature')
print('\n================ R2 HEADLINE ================')
st=se[se.strength=='strong']
print('strong features: raw ratio mean=%.3f, beta-corrected mean=%.3f'%(st['ratio'].mean(),st['ratio_betacorr'].mean()))
print('Grid-conditional bootstrap SE (separation-robust). beta-correction tests the 1/beta artifact.')
print('\nR2 complete.')


  bootstrap 50/200 (32s)


  bootstrap 100/200 (62s)


  bootstrap 150/200 (95s)


  bootstrap 200/200 (127s)
bootstrap done 127s

=== SE ratio (free gamma vs rank-one beta*WoE), bootstrap, per feature ===
                             feature strength   miss   beta  se_woe  se_gam  ratio  ratio_betacorr
RevolvingUtilizationOfUnsecuredLines   strong nomiss  0.680   0.023   0.028  1.181           0.803
             NumberOfTimes90DaysLate   strong nomiss  0.600   0.009   0.008  0.834           0.500
NumberOfTime30-59DaysPastDueNotWorse   strong nomiss  0.627   0.011   0.012  1.069           0.670
                                 age   strong nomiss  0.447   0.020   0.032  1.652           0.738
     NumberOfOpenCreditLinesAndLoans     weak nomiss -0.061   0.011   0.021  1.883           0.115
                       MonthlyIncome   strong   miss  0.361   0.014   0.030  2.098           0.757
                  NumberOfDependents     weak   miss  0.190   0.013   0.023  1.797           0.341

=== TOST equivalence (margin [0.80,1.25]) on per-feature SE ratio ===
  raw         


Saved: R2_se_ratio_box, R2_se_ratio_feature

================ R2 HEADLINE ================
strong features: raw ratio mean=1.367, beta-corrected mean=0.694
Grid-conditional bootstrap SE (separation-robust). beta-correction tests the 1/beta artifact.

R2 complete.
